# 04b — Alpha / gamma / density sensitivity analysis

Predefined robustness check on the Graphical LASSO model-selection strategy itself (Methods 2.11): (i) an EBIC grid across alpha and gamma showing that the density<=0.20 primary selection rule is not fragile, and (ii) bridge-biomarker rank stability across a neighborhood of alpha values around the selected solution (alpha=0.125).

## Environment setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-org>/<your-repo>.git"  # TODO: set to the published GitHub repo URL
REPO_DIR_NAME = "pcos_network_architecture_jcem_v3_jcem_10of10"


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if _in_colab():
    PROJECT_ROOT = Path("/content") / REPO_DIR_NAME
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "scikit-learn", "networkx", "statsmodels", "openpyxl"], check=True)
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not ((PROJECT_ROOT / "data_raw").exists() and (PROJECT_ROOT / "outputs").exists()):
        if PROJECT_ROOT.parent == PROJECT_ROOT:
            break
        PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts._pipeline_common import *  # noqa: F401,F403

print("Project root:", PROJECT_ROOT)


In [ ]:
import numpy as np, pandas as pd
from sklearn.covariance import GraphicalLasso

fd = pd.read_csv(PROJECT_ROOT / 'outputs' / '01_ingest_harmonize' / 'feature_dictionary_v2.csv')
primary = fd.loc[fd.included_primary, 'feature'].tolist()
Z = pd.read_csv(PROJECT_ROOT / 'outputs' / '02_primary_network_dataset' / 'primary_direct_features_z.csv')
cols = primary
n, p = Z.shape

alphas = [0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60]
gammas = [0.0, 0.25, 0.5, 0.75, 1.0]

# The Graphical LASSO fit does not depend on gamma (gamma only rescales the EBIC penalty), so each
# alpha is fit once and then evaluated across the full gamma grid.
fits = {}
for a in alphas:
    m = GraphicalLasso(alpha=a, max_iter=150, tol=1e-3).fit(Z)
    prec = m.precision_
    off = (np.abs(prec) > 1e-8)
    np.fill_diagonal(off, False)
    e = int(off.sum() / 2)
    density = e / (p * (p - 1) / 2)
    score = float(m.score(Z))
    fits[a] = {'model': m, 'n_edges': e, 'density': density, 'score': score}

grid_rows = []
for gamma in gammas:
    for a in alphas:
        f = fits[a]
        ebic = float(-2 * n * f['score'] + f['n_edges'] * np.log(n) + 4 * gamma * f['n_edges'] * np.log(p))
        grid_rows.append({
            'gamma': gamma, 'alpha': a, 'log_likelihood_per_sample': f['score'],
            'n_edges': f['n_edges'], 'density': f['density'], 'EBIC': ebic,
            'eligible_density_le_0_20': bool(f['density'] <= 0.20),
        })
grid = pd.DataFrame(grid_rows)
print(grid.head(11))


### Bridge-rank stability across neighboring alpha values

In [ ]:
bridge_alphas = [0.075, 0.10, 0.125, 0.15, 0.20, 0.25, 0.30]
bridge_rows = []
for a in bridge_alphas:
    f = fits[a]
    m = f['model']
    prec = m.precision_
    D = np.sqrt(np.diag(prec))
    P = -prec / np.outer(D, D)
    np.fill_diagonal(P, 1)
    edges = edge_df_from_partial(P, cols)
    edges['source_domain'] = edges.source.map(DOM)
    edges['target_domain'] = edges.target.map(DOM)
    metrics, _ = graph_metrics(edges, {c: DOM[c] for c in cols})
    top8 = metrics.sort_values('bridge_strength', ascending=False).head(8).reset_index(drop=True)
    for rank, row in enumerate(top8.itertuples(), start=1):
        bridge_rows.append({
            'alpha': a, 'n_edges': f['n_edges'], 'density': f['density'], 'rank': rank,
            'feature': row.feature, 'domain': row.domain, 'bridge_strength': row.bridge_strength,
        })
bridge_rank = pd.DataFrame(bridge_rows)

out_dir = PROJECT_ROOT / 'outputs' / '04_primary_graphical_lasso_sensitivity'
out_dir.mkdir(parents=True, exist_ok=True)
grid.to_csv(out_dir / 'alpha_gamma_density_selection_grid.csv', index=False)
bridge_rank.to_csv(out_dir / 'bridge_rank_sensitivity_by_alpha.csv', index=False)

bridge_rank.head(16)
